# 🔧 Predictive Maintenance using LSTM, RF, and XGBoost
This notebook demonstrates AI-driven predictive maintenance using NASA-style CMAPSS engine data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dropout, Dense
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import shap
import seaborn as sns

In [ ]:
df = pd.read_csv("../data/sample_engine_data.csv")
df.head()

In [ ]:
FEATURES = ['operational_setting_1', 'operational_setting_2',
            'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5']
scaler = MinMaxScaler()
df[FEATURES] = scaler.fit_transform(df[FEATURES])

In [ ]:
window_size = 10
X_seq, y_seq = [], []
for engine_id in df['engine_id'].unique():
    ed = df[df['engine_id'] == engine_id]
    for i in range(len(ed) - window_size):
        X_seq.append(ed[FEATURES].iloc[i:i+window_size].values)
        y_seq.append(ed['RUL'].iloc[i+window_size])
X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

In [ ]:
model = Sequential([
    LSTM(64, input_shape=(X_seq.shape[1], X_seq.shape[2])),
    Dropout(0.2),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')
history = model.fit(X_seq, y_seq, epochs=10, batch_size=16, validation_split=0.2)
model.save("../models/lstm_model.h5")

In [ ]:
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title("LSTM Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
X = df[FEATURES].values
y = df['RUL'].values
rf = RandomForestRegressor(n_estimators=100)
xgb = XGBRegressor(n_estimators=100)
rf.fit(X, y)
xgb.fit(X, y)
joblib.dump(rf, "../models/rf_model.pkl")
joblib.dump(xgb, "../models/xgb_model.pkl")

In [ ]:
def print_metrics(y_true, y_pred, name):
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)
    print(f"{name} -> RMSE: {rmse:.2f}, R²: {r2:.2f}")

print_metrics(y_seq, model.predict(X_seq).flatten(), "LSTM")
print_metrics(y, rf.predict(X), "Random Forest")
print_metrics(y, xgb.predict(X), "XGBoost")

In [ ]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X)
shap.summary_plot(shap_values, X, feature_names=FEATURES)